# Bayesian optimization of enzymatic PET hydrolysis: campaign, robustness, and experimental efficiency

**Companion computational analysis for JACS Au manuscript Sections 2.3–2.5 (equal-weight BHET yield–selectivity desirability).**

This notebook documents three related yet distinct analyses:

1. **Historical five-cycle campaign:** GP-based qLogEI/qPI recommendations, recorded measurements, and cycle-level best conditions (Figure 4a–c).
2. **Initialization robustness:** 20/50/100 starting measurements under random/space-filling initialization, 30 repetitions per configuration (Figure 4d–e).
3. **Experimental efficiency:** original 10,000-run random and sequential maximin space-filling screening baselines against the *recorded* 25-row BO campaign (manuscript Table 2).

**Read before running:** This is a *retrospective analysis*. Measured responses are used for lookup *only after* candidate selection. The historical BO record is the source of the manuscript's cycle claims; replaying BayBE recommendations may differ with dependency versions. Do **not** treat generated replays as the original experiment. Run from the repository root with `data/` alongside this notebook, or put this notebook at the repository root.

**Counting convention:** 25 recorded BO evaluation rows represent **18 unique conditions**. Screening baselines sample without replacement. Comparisons thus use **nominal additional evaluation rows**, not unique independent experiments. The 203 constrained data points are the shared historical starting dataset, excluded from the additional-budget comparisons.

## Reproducibility settings and input checks

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
if not DATA_DIR.exists() and (ROOT / 'JACSAu_BO_master' / 'data').exists():
    DATA_DIR = ROOT / 'JACSAu_BO_master' / 'data'
assert DATA_DIR.exists(), f'Data directory not found: {DATA_DIR}'
OUTPUT_ROOT = ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', DATA_DIR.resolve())
print('Outputs:', OUTPUT_ROOT.resolve())

---
# Part I — Historical Bayesian optimization campaign (manuscript Section 2.3)

The observed 25 evaluation rows, and not a newly generated stochastic replay, define the published trajectory. qLogEI and qPI proposal-generation code is retained as a transparent methods implementation. The optional reproduction call remains disabled by default.

## 1. Imports and campaign settings

The random seed, batch sizes, response bounds, and desirability weights are declared once. Changing the two weights allows the same workflow to represent alternative yield–selectivity priorities.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from baybe import Campaign
from baybe.acquisition import qLogExpectedImprovement, qProbabilityOfImprovement
from baybe.kernels.basic import MaternKernel
from baybe.objectives import DesirabilityObjective
from baybe.parameters import NumericalDiscreteParameter
from baybe.recommenders import BotorchRecommender
from baybe.searchspace import SearchSpace
from baybe.surrogates import GaussianProcessSurrogate
from baybe.targets import NumericalTarget
from baybe.utils import random as baybe_random


def locate_notebook_dir() -> Path:
    '''Locate the data folder when run from the notebook or project root.'''
    for candidate in [Path.cwd(), Path.cwd() / "bo_campaign_publication"]:
        if (candidate / "data").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate bo_campaign_publication/data")


NOTEBOOK_DIR = locate_notebook_dir()
DATA_DIR = NOTEBOOK_DIR / "data"

FEATURES = [
    "time_h",
    "substrate_g_L",
    "enzyme_nM",
    "EG_vol_percent",
    "temperature_C",
    "pH",
]
TARGETS = ["BHET_selectivity_percent", "BHET_yield_mg_g_PET"]

RANDOM_SEED = 42
N_CYCLES = 5
PROPOSALS_PER_ACQUISITION = 5
EXPERIMENTS_PER_CYCLE = 5

SELECTIVITY_BOUNDS = (0.0, 100.0)
YIELD_BOUNDS = (0.0, 12.910625)  # mg BHET g PET^-1
DESIRABILITY_WEIGHTS = (1.0, 1.0)

baybe_random.set_random_seed(RANDOM_SEED)

DATA_DIR = ROOT / "data"
OUTPUT_DIR = OUTPUT_ROOT / "historical_campaign"


## 2. Load the processed campaign data

The initial dataset contains 203 measured conditions. The candidate dataset contains the remaining 237 conditions available to the optimizer. Experimental responses in the candidate file are used only after a condition has been selected; they are not supplied to the surrogate during recommendation.

In [ ]:
initial_measurements = pd.read_csv(DATA_DIR / "initial_measurements.csv")
candidate_measurements = pd.read_csv(DATA_DIR / "candidate_measurements.csv")
recorded_pool = pd.read_csv(DATA_DIR / "recorded_candidate_pool.csv")
recorded_evaluations = pd.read_csv(DATA_DIR / "recorded_evaluations.csv")

assert len(initial_measurements) == 203
assert len(candidate_measurements) == 237
assert set(FEATURES + TARGETS).issubset(initial_measurements.columns)
assert set(FEATURES + TARGETS).issubset(candidate_measurements.columns)
assert not initial_measurements.duplicated(FEATURES).any()
assert not candidate_measurements.duplicated(FEATURES).any()

overlap = initial_measurements[FEATURES].merge(
    candidate_measurements[FEATURES], on=FEATURES, how="inner"
)
assert overlap.empty

display(
    pd.DataFrame(
        {
            "Dataset": ["Initial measurements", "Candidate conditions"],
            "Rows": [len(initial_measurements), len(candidate_measurements)],
            "BHET selectivity (%)": [
                f"{initial_measurements[TARGETS[0]].min():.2f}–{initial_measurements[TARGETS[0]].max():.2f}",
                f"{candidate_measurements[TARGETS[0]].min():.2f}–{candidate_measurements[TARGETS[0]].max():.2f}",
            ],
            "BHET yield (mg g$^{-1}$ PET)": [
                f"{initial_measurements[TARGETS[1]].min():.3f}–{initial_measurements[TARGETS[1]].max():.3f}",
                f"{candidate_measurements[TARGETS[1]].min():.3f}–{candidate_measurements[TARGETS[1]].max():.3f}",
            ],
        }
    )
)

## 3. Define the desirability objective

Selectivity and yield were transformed linearly to the interval from zero to one and then combined by an equal-weight arithmetic mean:

\[
D = \frac{1}{2}\left(\frac{S_{\mathrm{BHET}}}{100}
+ \frac{Y_{\mathrm{BHET}}}{12.910625}\right).
\]

The upper yield value is a normalization bound in mg BHET g PET$^{-1}$, not an observed concentration. The check below confirms that this equation reproduces the desirability values stored for all 25 campaign evaluations.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=DeprecationWarning)
    selectivity_target = NumericalTarget(
        name="BHET_selectivity_percent",
        mode="MAX",
        bounds=SELECTIVITY_BOUNDS,
        transformation="LINEAR",
    )
    yield_target = NumericalTarget(
        name="BHET_yield_mg_g_PET",
        mode="MAX",
        bounds=YIELD_BOUNDS,
        transformation="LINEAR",
    )

objective = DesirabilityObjective(
    targets=[selectivity_target, yield_target],
    weights=DESIRABILITY_WEIGHTS,
    scalarizer="MEAN",
)


def calculate_desirability(frame: pd.DataFrame) -> pd.Series:
    '''Return the equal-weight desirability in the manuscript units.'''
    return 0.5 * (
        frame["BHET_selectivity_percent"] / SELECTIVITY_BOUNDS[1]
        + frame["BHET_yield_mg_g_PET"] / YIELD_BOUNDS[1]
    )


recorded_check = pd.DataFrame(
    {
        "BHET_selectivity_percent": recorded_evaluations[
            "measured_selectivity_percent"
        ],
        "BHET_yield_mg_g_PET": recorded_evaluations[
            "measured_yield_mg_g_PET"
        ],
    }
)
maximum_error = np.max(
    np.abs(
        calculate_desirability(recorded_check)
        - recorded_evaluations["measured_desirability"]
    )
)
assert maximum_error < 1e-12
print(f"Maximum desirability verification error: {maximum_error:.2e}")

## 4. Construct the search space and surrogate models

All six experimental variables were treated as discrete numerical parameters. Both acquisition strategies used an independent Gaussian-process surrogate with a Matérn kernel. Each campaign was initialized with the same 203 measurements.

In [ ]:
all_conditions = pd.concat(
    [initial_measurements[FEATURES], candidate_measurements[FEATURES]],
    ignore_index=True,
)

parameters = [
    NumericalDiscreteParameter(name=name, values=all_conditions[name].unique())
    for name in FEATURES
]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    searchspace = SearchSpace.from_dataframe(
        df=all_conditions,
        parameters=parameters,
    )


def make_surrogate() -> GaussianProcessSurrogate:
    surrogate = GaussianProcessSurrogate()
    surrogate.kernel_factory = MaternKernel()
    return surrogate


def make_campaign(acquisition_function) -> Campaign:
    recommender = BotorchRecommender(
        sequential_continuous=True,
        surrogate_model=make_surrogate(),
        acquisition_function=acquisition_function,
    )
    campaign = Campaign(
        searchspace,
        objective,
        recommender,
        allow_recommending_already_recommended=False,
        allow_recommending_already_measured=False,
    )
    campaign.add_measurements(initial_measurements[FEATURES + TARGETS])
    # Fit once explicitly because posterior means are accessed immediately
    # after the first recommendation, as in the experimental campaign code.
    campaign.recommender._surrogate_model.fit(
        searchspace=searchspace,
        objective=objective,
        measurements=initial_measurements[FEATURES + TARGETS],
    )
    return campaign


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    campaign_qlogei = make_campaign(qLogExpectedImprovement())
    campaign_qpi = make_campaign(qProbabilityOfImprovement())
print("Two campaigns initialized with 203 measurements each.")

## 5. Generate and rank one recommendation batch

Each campaign proposes five conditions independently. Posterior means are converted to desirability, and the combined ten-row pool is sorted by predicted desirability. No fixed qLogEI:qPI allocation is imposed among the five selected rows.

The historical workflow did not remove a condition proposed by both acquisition functions before ranking. Such a condition could therefore appear twice in the selected batch. The code below preserves that behavior.

In [ ]:
def propose_with_posterior(campaign: Campaign, acquisition_label: str) -> pd.DataFrame:
    '''Generate five recommendations and attach posterior-mean predictions.'''
    recommendation = campaign.recommend(batch_size=PROPOSALS_PER_ACQUISITION).copy()
    posterior = campaign.recommender._surrogate_model.posterior(
        candidates=recommendation
    )
    prediction = posterior.mean.detach().cpu().numpy()

    recommendation.insert(0, "acquisition", acquisition_label)
    recommendation["predicted_selectivity_percent"] = prediction[:, 0]
    recommendation["predicted_yield_mg_g_PET"] = prediction[:, 1]

    predicted_targets = pd.DataFrame(
        {
            "BHET_selectivity_percent": recommendation[
                "predicted_selectivity_percent"
            ],
            "BHET_yield_mg_g_PET": recommendation[
                "predicted_yield_mg_g_PET"
            ],
        }
    )
    recommendation["predicted_desirability"] = objective.transform(
        predicted_targets
    ).to_numpy()
    return recommendation


def recommend_experimental_batch(
    campaign_qlogei: Campaign,
    campaign_qpi: Campaign,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Pool ten proposals and select the five highest predicted scores.'''
    pool = pd.concat(
        [
            propose_with_posterior(campaign_qlogei, "qLogEI"),
            propose_with_posterior(campaign_qpi, "qPI"),
        ],
        ignore_index=True,
    )
    ranked_pool = pool.sort_values(
        "predicted_desirability",
        ascending=False,
        kind="mergesort",
    ).reset_index(drop=True)
    selected_batch = ranked_pool.head(EXPERIMENTS_PER_CYCLE).copy()
    return ranked_pool, selected_batch

## 6. Add the experimental measurements

After the selected conditions were evaluated experimentally, their measured selectivity and yield were added to both campaigns. The next cycle was therefore based on the same accumulated measurements for qLogEI and qPI.

In [ ]:
def add_experimental_results(
    measured_batch: pd.DataFrame,
    campaign_qlogei: Campaign,
    campaign_qpi: Campaign,
) -> None:
    '''Update both campaigns with the same newly measured conditions.'''
    required = FEATURES + TARGETS
    missing = set(required) - set(measured_batch.columns)
    if missing:
        raise ValueError(f"Missing measurement columns: {sorted(missing)}")

    for campaign in (campaign_qlogei, campaign_qpi):
        campaign.add_measurements(measured_batch[required])


def lookup_measured_outcomes(selected_conditions: pd.DataFrame) -> pd.DataFrame:
    '''Offline substitute for the laboratory measurement step.

    This lookup is useful when rerunning the historical, fully measured design
    space. During the live campaign, this function was replaced by the newly
    returned experimental measurements.
    '''
    return selected_conditions[FEATURES].merge(
        candidate_measurements[FEATURES + TARGETS],
        on=FEATURES,
        how="left",
        validate="many_to_one",
    )

## 7. Five-cycle campaign loop

The function below collects the complete workflow. For a live campaign, the selected conditions are exported for experimental evaluation and the measured results are supplied to `add_experimental_results`. The optional lookup shown here permits retrospective execution because all candidate outcomes are now available.

In [ ]:
def run_campaign(n_cycles: int = N_CYCLES):
    '''Run the BO workflow against the retrospectively measured candidate set.'''
    baybe_random.set_random_seed(RANDOM_SEED)
    qlogei = make_campaign(qLogExpectedImprovement())
    qpi = make_campaign(qProbabilityOfImprovement())

    proposal_history = []
    evaluation_history = []

    for cycle in range(1, n_cycles + 1):
        ranked_pool, selected_conditions = recommend_experimental_batch(qlogei, qpi)
        ranked_pool.insert(0, "cycle", cycle)

        measured_batch = lookup_measured_outcomes(selected_conditions)
        if measured_batch[TARGETS].isna().any().any():
            raise ValueError(f"Missing candidate outcome in cycle {cycle}")

        measured_batch["measured_desirability"] = calculate_desirability(
            measured_batch
        )
        measured_batch.insert(0, "cycle", cycle)
        add_experimental_results(measured_batch, qlogei, qpi)

        proposal_history.append(ranked_pool)
        evaluation_history.append(measured_batch)

    return (
        pd.concat(proposal_history, ignore_index=True),
        pd.concat(evaluation_history, ignore_index=True),
    )


# Uncomment to generate a new retrospective trajectory with the installed
# software versions. The recorded campaign itself is summarized below.
# generated_pool, generated_evaluations = run_campaign()

## 8. Recorded campaign outcome

The tables below summarize the 25 evaluation rows recorded during the five experimental cycles. The qLogEI:qPI composition was an outcome of desirability ranking, not a preassigned quota. Eighteen unique conditions were evaluated because seven rows were cross-acquisition duplicates.

In [ ]:
acquisition_by_cycle = (
    recorded_evaluations.groupby(["cycle", "acquisition"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["qLogEI", "qPI"], fill_value=0)
)

unique_conditions = len(recorded_evaluations.drop_duplicates(FEATURES))
duplicate_rows = len(recorded_evaluations) - unique_conditions
best_by_cycle = recorded_evaluations.groupby("cycle")[
    "measured_desirability"
].max().cummax()

display(acquisition_by_cycle.rename_axis(index="BO cycle"))
display(
    pd.Series(
        {
            "Evaluation rows": len(recorded_evaluations),
            "Unique experimental conditions": unique_conditions,
            "Cross-acquisition duplicate rows": duplicate_rows,
            "Final best desirability": best_by_cycle.iloc[-1],
        },
        name="Recorded campaign",
    ).to_frame()
)

fig, ax = plt.subplots(figsize=(5.8, 3.6))
ax.plot(best_by_cycle.index, best_by_cycle.values, "o-", color="#2F6B9A")
ax.set(
    xlabel="BO cycle",
    ylabel="Best measured desirability",
    xticks=range(1, N_CYCLES + 1),
    title="Recorded campaign performance",
)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 9. Implementation summary

The campaign used two independent Gaussian-process models with the same initial measurements, Matérn kernel, and equal-weight desirability objective. qLogEI and qPI each proposed five candidates per cycle. The ten proposals were ranked together, and five rows were selected without imposing an acquisition-function quota or removing cross-acquisition duplicates. Both models were updated with the measured outcomes before the next cycle.

The files in `data/recorded_candidate_pool.csv` and `data/recorded_evaluations.csv` preserve the recommendations and measurements generated during the campaign in the units reported in the manuscript.

### Manuscript-facing audit: best recorded response per cycle

The per-cycle best desirability is distinct from (a) best yield, (b) best selectivity, and (c) cumulative best-so-far. Export the comparison table for figure/caption checking.

In [ ]:
cycle_detail = recorded_evaluations.loc[recorded_evaluations.groupby('cycle')['measured_desirability'].idxmax(),
    ['cycle'] + FEATURES + ['measured_yield_mg_g_PET','measured_selectivity_percent','measured_desirability']].sort_values('cycle').reset_index(drop=True)
cycle_detail['best_so_far_D'] = cycle_detail['measured_desirability'].cummax()
cycle_detail['cycle_max_yield'] = recorded_evaluations.groupby('cycle')['measured_yield_mg_g_PET'].max().to_numpy()
cycle_detail['cycle_max_selectivity'] = recorded_evaluations.groupby('cycle')['measured_selectivity_percent'].max().to_numpy()
cycle_detail['cycle_mean_selectivity'] = recorded_evaluations.groupby('cycle')['measured_selectivity_percent'].mean().to_numpy()
display(cycle_detail)
(OUTPUT_ROOT/'historical_campaign').mkdir(parents=True,exist_ok=True)
cycle_detail.to_csv(OUTPUT_ROOT/'historical_campaign'/'cycle_by_cycle_audit.csv',index=False)
assert len(cycle_detail)==5
assert abs(cycle_detail.iloc[-1].measured_desirability-0.739064731)<1e-6


---
# Part II — Initialization robustness (manuscript Section 2.4)

**Scientific question:** How do the number and spatial distribution of initial measurements change the reliability of five-cycle BO? The original robustness implementation follows; it is not the independent standalone space-filling screening method in Part III. Running all 180 BO trajectories requires a compatible BayBE/BoTorch installation and may take substantial time.

## 1. Imports and paths

Required packages are `numpy`, `pandas`, `matplotlib`, `joblib`, `baybe`, and `botorch`. Run this notebook from the supplied publication folder, which contains `data/full_440_measurements.csv`.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import pickle
import time
import warnings

from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def locate_notebook_folder():
    for candidate in [Path.cwd(), Path.cwd() / "bo_campaign_publication"]:
        if (candidate / "data" / "full_440_measurements.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate data/full_440_measurements.csv")


NOTEBOOK_DIR = locate_notebook_folder()
DATA_PATH = NOTEBOOK_DIR / "data" / "full_440_measurements.csv"
RESULTS_DIR = NOTEBOOK_DIR / "robustness_results_self_contained"
RUN_DIR = RESULTS_DIR / "individual_runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "full_440_measurements.csv"
RESULTS_DIR = OUTPUT_ROOT / "initialization_robustness"
RUN_DIR = RESULTS_DIR / "individual_runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)


## 2. Analysis settings

In [ ]:
FEATURES = [
    "time_h", "substrate_g_L", "enzyme_nM", "EG_vol_percent",
    "temperature_C", "pH",
]
TARGETS = ["BHET_selectivity_percent", "BHET_yield_mg_g_PET"]
SELECTIVITY_BOUNDS = (0.0, 100.0)
YIELD_BOUNDS = (0.0, 12.910625)  # mg BHET g PET^-1
DESIRABILITY_WEIGHTS = (1.0, 1.0)

RANDOM_INITIAL_SIZES = [20, 50, 100]
SPACE_FILLING_INITIAL_SIZES = [20, 50, 100]
N_REPETITIONS = 30
N_CYCLES = 5
PROPOSALS_PER_ACQUISITION = 5
EXPERIMENTS_PER_CYCLE = 5
BASE_SEED = 42
N_PARALLEL_JOBS = 4

# Set a switch to False to load existing runs without calculating missing ones.
RUN_RANDOM_STUDY = True
RUN_SPACE_FILLING_STUDY = True

DATA_PATH = DATA_DIR / "full_440_measurements.csv"
RESULTS_DIR = OUTPUT_ROOT / "initialization_robustness"


## 3. Load the 440-condition measured design space

For each simulated campaign, the selected initial rows are treated as observed and all remaining rows constitute the candidate pool. Thus, initial sizes of 20, 50, and 100 leave 420, 390, and 340 candidate conditions, respectively.

The equal-weight desirability is

\[
D=\frac{1}{2}\left(\frac{S_{\mathrm{BHET}}}{100}+
\frac{Y_{\mathrm{BHET}}}{12.910625}\right).
\]

In [ ]:
full_data = pd.read_csv(DATA_PATH)
assert len(full_data) == 440
assert not full_data.duplicated(FEATURES).any()
assert np.isfinite(full_data[FEATURES + TARGETS].to_numpy(float)).all()

full_data["desirability"] = 0.5 * (
    full_data[TARGETS[0]] / 100.0
    + full_data[TARGETS[1]] / 12.910625
)
global_best = float(full_data["desirability"].max())
top5_threshold = float(full_data["desirability"].quantile(0.95))

display(pd.DataFrame({
    "initial_size": [20, 50, 100],
    "initial_candidate_pool": [420, 390, 340],
}))

## 4. Select the initial observations

Random subsets are sampled uniformly without replacement. Space-filling subsets are produced by randomized-start greedy maximin selection after min–max scaling the six experimental variables. Neither method uses yield or selectivity when choosing the initial rows.

In [ ]:
@dataclass(frozen=True)
class RunSpec:
    method: str
    initial_size: int
    replicate: int
    seed: int

    @property
    def run_id(self) -> str:
        return f"{self.method}_n{self.initial_size}_r{self.replicate:02d}"


def desirability(frame: pd.DataFrame) -> pd.Series:
    return 0.5 * (
        frame[TARGETS[0]] / SELECTIVITY_BOUNDS[1]
        + frame[TARGETS[1]] / YIELD_BOUNDS[1]
    )


def scale_features(frame: pd.DataFrame) -> np.ndarray:
    values = frame[FEATURES].to_numpy(dtype=float)
    lower = values.min(axis=0)
    span = values.max(axis=0) - lower
    span[span == 0.0] = 1.0
    return (values - lower) / span


def random_subset(n_rows: int, size: int, rng: np.random.Generator) -> np.ndarray:
    return np.sort(rng.choice(n_rows, size=size, replace=False))


def maximin_subset(
    scaled_features: np.ndarray,
    size: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Select observed rows using randomized-start greedy maximin sampling."""
    n_rows = len(scaled_features)
    first = int(rng.integers(n_rows))
    selected = [first]
    available = np.ones(n_rows, dtype=bool)
    available[first] = False

    squared_distance = np.sum(
        (scaled_features - scaled_features[first]) ** 2,
        axis=1,
    )
    squared_distance[first] = -np.inf

    while len(selected) < size:
        best = np.max(squared_distance[available])
        ties = np.flatnonzero(available & np.isclose(squared_distance, best))
        next_index = int(rng.choice(ties))
        selected.append(next_index)
        available[next_index] = False

        distance_to_new = np.sum(
            (scaled_features - scaled_features[next_index]) ** 2,
            axis=1,
        )
        squared_distance = np.minimum(squared_distance, distance_to_new)
        squared_distance[~available] = -np.inf

    return np.sort(np.asarray(selected, dtype=int))


def minimum_pairwise_distance(scaled: np.ndarray) -> float:
    if len(scaled) < 2:
        return np.nan
    squared = np.sum((scaled[:, None, :] - scaled[None, :, :]) ** 2, axis=2)
    np.fill_diagonal(squared, np.inf)
    return float(np.sqrt(np.min(squared)))

## 5. Construct and fit the Gaussian processes

The code below explicitly creates a Matérn-kernel `GaussianProcessSurrogate`, supplies only `initial[FEATURES + TARGETS]` as measurements, and fits the GP to those rows. Separate campaigns use qLog expected improvement (qLogEI) and q probability of improvement (qPI).

`allow_recommending_already_measured=False` prevents the initial observations and subsequent GP measurements from being proposed again.

In [ ]:
def make_baybe_components(full_data: pd.DataFrame, initial: pd.DataFrame):
    from baybe import Campaign
    from baybe.acquisition import qLogExpectedImprovement, qProbabilityOfImprovement
    from baybe.kernels.basic import MaternKernel
    from baybe.objectives import DesirabilityObjective
    from baybe.parameters import NumericalDiscreteParameter
    from baybe.recommenders import BotorchRecommender
    from baybe.searchspace import SearchSpace
    from baybe.surrogates import GaussianProcessSurrogate
    from baybe.targets import NumericalTarget

    selectivity_target = NumericalTarget(
        name=TARGETS[0],
        mode="MAX",
        bounds=SELECTIVITY_BOUNDS,
        transformation="LINEAR",
    )
    yield_target = NumericalTarget(
        name=TARGETS[1],
        mode="MAX",
        bounds=YIELD_BOUNDS,
        transformation="LINEAR",
    )
    objective = DesirabilityObjective(
        targets=[selectivity_target, yield_target],
        weights=DESIRABILITY_WEIGHTS,
        scalarizer="MEAN",
    )

    parameters = [
        NumericalDiscreteParameter(name=name, values=full_data[name].unique())
        for name in FEATURES
    ]
    searchspace = SearchSpace.from_dataframe(
        df=full_data[FEATURES],
        parameters=parameters,
    )

    def campaign(acquisition):
        surrogate = GaussianProcessSurrogate()
        surrogate.kernel_factory = MaternKernel()
        recommender = BotorchRecommender(
            sequential_continuous=True,
            surrogate_model=surrogate,
            acquisition_function=acquisition,
        )
        result = Campaign(
            searchspace,
            objective,
            recommender,
            allow_recommending_already_recommended=False,
            allow_recommending_already_measured=False,
        )
        result.add_measurements(initial[FEATURES + TARGETS])
        result.recommender._surrogate_model.fit(
            searchspace=searchspace,
            objective=objective,
            measurements=initial[FEATURES + TARGETS],
        )
        return result

    return (
        objective,
        campaign(qLogExpectedImprovement()),
        campaign(qProbabilityOfImprovement()),
    )


def condition_keys(frame: pd.DataFrame) -> pd.Series:
    return frame[FEATURES].round(10).astype(str).agg("|".join, axis=1)


def metric_row(
    spec: RunSpec,
    strategy: str,
    cycle: int,
    initial_best: float,
    best: float,
    global_best: float,
    top5_threshold: float,
    unique_evaluated: int,
    evaluation_rows: int,
) -> dict:
    return {
        "run_id": spec.run_id,
        "initialization": spec.method,
        "initial_size": spec.initial_size,
        "replicate": spec.replicate,
        "seed": spec.seed,
        "strategy": strategy,
        "cycle": cycle,
        "evaluation_rows": evaluation_rows,
        "unique_evaluated_conditions": unique_evaluated,
        "duplicate_evaluation_rows": evaluation_rows - unique_evaluated,
        "initial_best_desirability": initial_best,
        "best_desirability": best,
        "improvement_from_initial": best - initial_best,
        "simple_regret": global_best - best,
        "top5_percent_found": int(best >= top5_threshold - 1e-12),
        "global_best_found": int(best >= global_best - 1e-12),
    }


def propose(campaign, objective, acquisition_label: str) -> pd.DataFrame:
    recommendation = campaign.recommend(
        batch_size=PROPOSALS_PER_ACQUISITION
    ).copy()
    posterior = campaign.recommender._surrogate_model.posterior(
        candidates=recommendation
    )
    prediction = posterior.mean.detach().cpu().numpy()
    recommendation.insert(0, "acquisition", acquisition_label)
    recommendation["predicted_selectivity_percent"] = prediction[:, 0]
    recommendation["predicted_yield_mg_g_PET"] = prediction[:, 1]
    predicted_targets = pd.DataFrame(
        {
            TARGETS[0]: prediction[:, 0],
            TARGETS[1]: prediction[:, 1],
        }
    )
    recommendation["predicted_desirability"] = objective.transform(
        predicted_targets
    ).to_numpy()
    return recommendation

## 6. Run one complete campaign

The crucial split is visible near the beginning of `run_one`: `initial` contains the selected 20, 50, or 100 rows, whereas `available` contains only the other 420, 390, or 340 rows. BO recommends conditions before their outcomes are merged from `available`.

After each cycle, the selected unique conditions are added to both GP campaigns. Cross-acquisition duplicates still consume evaluation rows, reproducing the historical selection rule, but one stored mean is added only once to the retrospective GP update.

In [ ]:
def run_one(spec: RunSpec, data_path: str) -> dict[str, list[dict]]:
    from baybe.utils import random as baybe_random
    from botorch.exceptions.errors import ModelFittingError

    started = time.perf_counter()
    full_data = pd.read_csv(data_path)
    full_data["desirability"] = desirability(full_data)
    scaled = scale_features(full_data)
    rng = np.random.default_rng(spec.seed)

    if spec.method == "random":
        initial_indices = random_subset(len(full_data), spec.initial_size, rng)
    elif spec.method == "space_filling":
        initial_indices = maximin_subset(scaled, spec.initial_size, rng)
    else:
        raise ValueError(f"Unknown initialization method: {spec.method}")

    initial = full_data.iloc[initial_indices].copy()
    candidate_mask = np.ones(len(full_data), dtype=bool)
    candidate_mask[initial_indices] = False
    available = full_data.loc[candidate_mask].copy()

    global_best = float(full_data["desirability"].max())
    top5_threshold = float(full_data["desirability"].quantile(0.95))
    initial_best = float(initial["desirability"].max())
    initial_keys = set(condition_keys(initial))

    initial_rows = [
        {
            "run_id": spec.run_id,
            "initialization": spec.method,
            "initial_size": spec.initial_size,
            "replicate": spec.replicate,
            "seed": spec.seed,
            "minimum_scaled_pairwise_distance": minimum_pairwise_distance(
                scaled[initial_indices]
            ),
            **row,
        }
        for row in initial[FEATURES + TARGETS + ["desirability"]].to_dict("records")
    ]

    metrics = [
        metric_row(
            spec,
            "BO",
            0,
            initial_best,
            initial_best,
            global_best,
            top5_threshold,
            0,
            0,
        ),
        metric_row(
            spec,
            "random_search",
            0,
            initial_best,
            initial_best,
            global_best,
            top5_threshold,
            0,
            0,
        ),
    ]

    # Rare GP hyperparameter fits can fail for a particular numerical start.
    # The initial subset remains fixed while the full BO trajectory is retried
    # with a deterministic alternative BayBE seed.
    max_fitting_attempts = 5
    for fitting_attempt in range(max_fitting_attempts):
        proposal_rows = []
        selected_rows = []
        attempt_metrics = []
        bo_seen: set[str] = set()
        numerical_seed = spec.seed + fitting_attempt * 1_000_000
        baybe_random.set_random_seed(numerical_seed)

        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                objective, qlogei, qpi = make_baybe_components(full_data, initial)

                for cycle in range(1, N_CYCLES + 1):
                    pooled = pd.concat(
                        [
                            propose(qlogei, objective, "qLogEI"),
                            propose(qpi, objective, "qPI"),
                        ],
                        ignore_index=True,
                    )
                    pooled = pooled.sort_values(
                        "predicted_desirability",
                        ascending=False,
                        kind="mergesort",
                    ).reset_index(drop=True)
                    pooled["selected"] = pooled.index < EXPERIMENTS_PER_CYCLE

                    measured_pool = pooled.merge(
                        available[FEATURES + TARGETS + ["desirability"]],
                        on=FEATURES,
                        how="left",
                        validate="many_to_one",
                    )
                    if measured_pool[TARGETS].isna().any().any():
                        raise RuntimeError(
                            f"Outcome lookup failed for {spec.run_id}, cycle {cycle}"
                        )

                    for rank, row in measured_pool.iterrows():
                        proposal_rows.append(
                            {
                                "run_id": spec.run_id,
                                "initialization": spec.method,
                                "initial_size": spec.initial_size,
                                "replicate": spec.replicate,
                                "seed": spec.seed,
                                "numerical_seed": numerical_seed,
                                "cycle": cycle,
                                "rank": rank + 1,
                                **row.to_dict(),
                            }
                        )

                    selected = measured_pool.head(EXPERIMENTS_PER_CYCLE).copy()
                    for rank, row in selected.iterrows():
                        selected_rows.append(
                            {
                                "run_id": spec.run_id,
                                "initialization": spec.method,
                                "initial_size": spec.initial_size,
                                "replicate": spec.replicate,
                                "seed": spec.seed,
                                "numerical_seed": numerical_seed,
                                "cycle": cycle,
                                "rank": rank + 1,
                                **row.to_dict(),
                            }
                        )

                    # A cross-acquisition duplicate consumes an evaluation row,
                    # but the retrospective table contains only one stored mean
                    # for that condition. Add it once to avoid treating the same
                    # lookup value as an independent experimental replicate.
                    unique_update = selected.drop_duplicates(FEATURES)
                    qlogei.add_measurements(unique_update[FEATURES + TARGETS])
                    qpi.add_measurements(unique_update[FEATURES + TARGETS])

                    bo_seen.update(condition_keys(selected))
                    best = max(
                        initial_best,
                        max(row["desirability"] for row in selected_rows),
                    )
                    attempt_metrics.append(
                        metric_row(
                            spec,
                            "BO",
                            cycle,
                            initial_best,
                            float(best),
                            global_best,
                            top5_threshold,
                            len(bo_seen),
                            cycle * EXPERIMENTS_PER_CYCLE,
                        )
                    )
            metrics.extend(attempt_metrics)
            break
        except ModelFittingError:
            if fitting_attempt == max_fitting_attempts - 1:
                raise

    # Paired random search uses the same initial subset and a separate seed.
    baseline_rng = np.random.default_rng(spec.seed + 1_000_003)
    remaining_indices = np.flatnonzero(candidate_mask)
    random_order = baseline_rng.choice(
        remaining_indices,
        size=N_CYCLES * EXPERIMENTS_PER_CYCLE,
        replace=False,
    )
    random_seen: set[str] = set()
    random_best = initial_best
    for cycle in range(1, N_CYCLES + 1):
        cycle_indices = random_order[
            (cycle - 1) * EXPERIMENTS_PER_CYCLE : cycle * EXPERIMENTS_PER_CYCLE
        ]
        batch = full_data.iloc[cycle_indices]
        random_seen.update(condition_keys(batch))
        random_best = max(random_best, float(batch["desirability"].max()))
        metrics.append(
            metric_row(
                spec,
                "random_search",
                cycle,
                initial_best,
                random_best,
                global_best,
                top5_threshold,
                len(random_seen),
                cycle * EXPERIMENTS_PER_CYCLE,
            )
        )

    return {
        "metrics": metrics,
        "initial_rows": initial_rows,
        "proposal_rows": proposal_rows,
        "selected_rows": selected_rows,
        "completed": [
            {
                "run_id": spec.run_id,
                "initialization": spec.method,
                "initial_size": spec.initial_size,
                "replicate": spec.replicate,
                "seed": spec.seed,
                "fitting_retries": fitting_attempt,
                "numerical_seed": numerical_seed,
                "runtime_seconds": time.perf_counter() - started,
            }
        ],
    }

## 7. Resume-safe execution

Each completed campaign is saved separately. If execution is interrupted, running the cells again calculates only missing runs. `joblib` provides parallel execution without hiding the scientific procedure in an external script.

In [ ]:
def build_specs(initialization, sizes):
    method_offset = 0 if initialization == "random" else 100_000
    return [
        RunSpec(
            method=initialization,
            initial_size=size,
            replicate=repetition,
            seed=BASE_SEED + method_offset + size * 100 + repetition,
        )
        for size in sizes
        for repetition in range(1, N_REPETITIONS + 1)
    ]


def run_and_save(spec):
    destination = RUN_DIR / f"{spec.run_id}.pkl"
    if destination.is_file():
        return spec.run_id
    result = run_one(spec, str(DATA_PATH))
    with destination.open("wb") as handle:
        pickle.dump(result, handle)
    return spec.run_id


def run_study(initialization, sizes, enabled):
    specs = build_specs(initialization, sizes)
    pending = [
        spec for spec in specs
        if not (RUN_DIR / f"{spec.run_id}.pkl").is_file()
    ]
    print(f"{initialization}: {len(specs)} requested, {len(pending)} pending")
    if enabled and pending:
        Parallel(n_jobs=N_PARALLEL_JOBS, verbose=10)(
            delayed(run_and_save)(spec) for spec in pending
        )
    elif not enabled:
        print("Execution disabled; existing results were left unchanged.")

# Part A — Random initial datasets

This cell performs the three random-initialization scenarios separately from the space-filling analysis: 30 repetitions each for initial sizes 20, 50, and 100.

In [ ]:
run_study(
    initialization="random",
    sizes=RANDOM_INITIAL_SIZES,
    enabled=RUN_RANDOM_STUDY,
)

# Part B — Space-filling initial datasets

This cell performs the three space-filling scenarios: 30 randomized-start maximin subsets each for initial sizes 20, 50, and 100.

In [ ]:
run_study(
    initialization="space_filling",
    sizes=SPACE_FILLING_INITIAL_SIZES,
    enabled=RUN_SPACE_FILLING_STUDY,
)

## 8. Combine the completed campaigns

The isolated results are combined into transparent tables containing the initial subsets, all ten proposals per cycle, the five selected evaluation rows, and cycle-level performance metrics.

In [ ]:
all_specs = (
    build_specs("random", RANDOM_INITIAL_SIZES)
    + build_specs("space_filling", SPACE_FILLING_INITIAL_SIZES)
)
result_files = [RUN_DIR / f"{spec.run_id}.pkl" for spec in all_specs]
available_files = [path for path in result_files if path.is_file()]
print(f"Completed campaign files: {len(available_files)}/{len(all_specs)}")

loaded = []
for path in available_files:
    with path.open("rb") as handle:
        loaded.append(pickle.load(handle))

assert loaded, "No completed campaigns were found."
metrics = pd.DataFrame(
    [row for item in loaded for row in item["metrics"]]
)
initial_subsets = pd.DataFrame(
    [row for item in loaded for row in item["initial_rows"]]
)
proposal_pools = pd.DataFrame(
    [row for item in loaded for row in item["proposal_rows"]]
)
selected_rows = pd.DataFrame(
    [row for item in loaded for row in item["selected_rows"]]
)
completed_runs = pd.DataFrame(
    [row for item in loaded for row in item["completed"]]
)

## 9. Verify the initial training sets and held-back candidates

These assertions provide an auditable confirmation that every GP starts with its declared initial size, all selected conditions belong to the finite 440-row space, and no selected BO condition was part of the same run's initial subset.

In [ ]:
assert completed_runs["run_id"].nunique() == 180

setting_counts = completed_runs.groupby(
    ["initialization", "initial_size"]
)["run_id"].nunique().unstack()
assert (setting_counts == N_REPETITIONS).all().all()

# The number of saved initial rows must equal the requested GP training size.
observed_sizes = initial_subsets.groupby("run_id").size()
declared_sizes = completed_runs.set_index("run_id")["initial_size"]
assert observed_sizes.equals(declared_sizes.loc[observed_sizes.index])

# All BO-selected rows must come from the 440-condition design space.
design_keys = set(condition_keys(full_data))
assert set(condition_keys(selected_rows)).issubset(design_keys)

# Within a run, no BO-selected row may overlap its initial GP training subset.
initial_keys = initial_subsets[["run_id"]].copy()
initial_keys["condition_key"] = condition_keys(initial_subsets)
selected_keys = selected_rows[["run_id"]].copy()
selected_keys["condition_key"] = condition_keys(selected_rows)
overlap = initial_keys.merge(
    selected_keys, on=["run_id", "condition_key"], how="inner"
)
assert overlap.empty

assert len(selected_rows) == (
    180 * N_CYCLES * EXPERIMENTS_PER_CYCLE
)

audit_table = pd.DataFrame({
    "initial_size": [20, 50, 100],
    "GP_training_rows_at_cycle_1": [20, 50, 100],
    "held_back_candidates_at_cycle_1": [420, 390, 340],
})
display(setting_counts.rename_axis(index="Initialization", columns="Initial size"))
display(audit_table)
print("All initialization, candidate-pool, and completeness checks passed.")

## 10. Final-cycle performance distributions

In [ ]:
final = metrics.loc[metrics["cycle"] == N_CYCLES].copy()
final_summary = (
    final.groupby(["initialization", "initial_size", "strategy"])
    .agg(
        repetitions=("run_id", "nunique"),
        median_best=("best_desirability", "median"),
        q25_best=("best_desirability", lambda x: x.quantile(0.25)),
        q75_best=("best_desirability", lambda x: x.quantile(0.75)),
        median_improvement=("improvement_from_initial", "median"),
        median_regret=("simple_regret", "median"),
        top5_success_percent=("top5_percent_found", lambda x: 100*x.mean()),
        global_best_success_percent=("global_best_found", lambda x: 100*x.mean()),
        median_unique_evaluations=("unique_evaluated_conditions", "median"),
        median_duplicate_rows=("duplicate_evaluation_rows", "median"),
    )
    .reset_index()
)
display(final_summary.round(4))

## 11. Best-so-far trajectories

Lines show medians across 30 repetitions and shaded bands show interquartile ranges. The red dotted line is the best desirability among all 440 measured conditions.

In [ ]:
method_titles = {"random": "Random", "space_filling": "Space-filling"}
styles = {
    "BO": ("#2F6B9A", "Bayesian optimization"),
    "random_search": ("#777777", "Random search"),
}

fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for row, method in enumerate(["random", "space_filling"]):
    for col, size in enumerate([20, 50, 100]):
        ax = axes[row, col]
        subset = metrics.loc[
            (metrics["initialization"] == method)
            & (metrics["initial_size"] == size)
        ]
        for strategy, (color, label) in styles.items():
            trajectory = subset.loc[subset["strategy"] == strategy].groupby(
                "cycle"
            )["best_desirability"].agg(
                median="median",
                q25=lambda x: x.quantile(0.25),
                q75=lambda x: x.quantile(0.75),
            )
            x = trajectory.index.to_numpy()
            ax.plot(x, trajectory["median"], marker="o", color=color, label=label)
            ax.fill_between(
                x, trajectory["q25"], trajectory["q75"], color=color, alpha=0.18
            )
        ax.axhline(global_best, color="#B24A3A", linestyle=":", linewidth=1)
        ax.set_title(f"{method_titles[method]}, n = {size}")
        ax.set_xticks(range(N_CYCLES + 1))
        ax.spines[["top", "right"]].set_visible(False)
        if row == 1:
            ax.set_xlabel("BO cycle")
        if col == 0:
            ax.set_ylabel("Best-so-far desirability")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.suptitle("Optimization performance across initial datasets", y=1.01)
fig.tight_layout()
plt.show()

## 12. Save publication and audit tables

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics.to_csv(RESULTS_DIR / "run_metrics.csv", index=False)
initial_subsets.to_csv(RESULTS_DIR / "initial_subsets.csv", index=False)
proposal_pools.to_csv(RESULTS_DIR / "bo_proposal_pools.csv", index=False)
selected_rows.to_csv(RESULTS_DIR / "bo_selected_rows.csv", index=False)
completed_runs.to_csv(RESULTS_DIR / "completed_runs.csv", index=False)
final_summary.to_csv(RESULTS_DIR / "final_summary.csv", index=False)
print(f"Results saved in: {RESULTS_DIR}")

## 13. Interpretation

This robustness study is distinct from experimental-efficiency analysis. Experimental efficiency asks how performance changes with evaluation count. Here, the five-cycle evaluation budget is fixed while the size and composition of the initial dataset are varied.

Random and space-filling results should be interpreted separately. For each initial size, report the distribution of final best-so-far desirability, improvement from the initial subset, regret to the full-data optimum, and the probability of finding a top-5% condition.

---
# Part III — Original experimental-efficiency comparison (manuscript Section 2.5 and Table 2)

**Important:** This section preserves the original `Experimental_efficiency_analysis.ipynb` random-order and sequential maximin algorithms (10,000 Monte Carlo campaigns each; seed `20260911`; 25-evaluation benchmark). The **only substantive adaptation** is loading the same measured conditions from the supplied publication CSVs rather than Excel sheets with alternate headings. Recorded BO results are not retrained.

**Space filling here is a standalone screening baseline**, not space-filling selection of the BO initial data in Part II.

## 1. Packages and reproducibility

Required packages: `numpy`, `pandas`, and `openpyxl`. If needed, install them in your Jupyter environment with `%pip install numpy pandas openpyxl` in a separate cell. Optional plots use `matplotlib`.

The seed makes results reproducible. Separate random-number streams are used for random and space-filling sampling.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

N_CAMPAIGNS = 10_000
SEED = 20260911
BUDGET = 25
TARGET_TOLERANCE = 1e-12  # Numerical precision only; not an experimental equivalence margin.

### Reproducible random numbers
This small helper gives each strategy its own repeatable random stream.

In [ ]:
def baseline_rng(which):
    # A fresh stream allows each analysis cell to be rerun reproducibly.
    return np.random.default_rng(np.random.SeedSequence(SEED).spawn(2)[which])

print(f"Campaigns per baseline: {N_CAMPAIGNS:,}; evaluation budget: {BUDGET}; seed: {SEED}")

## 2. EXCEL INPUTS — files beside this notebook

**Candidate file:** 237 conditions with six experimental inputs and measured selectivity/yield.

**BO file:** 25 evaluation rows, with cycle and acquisition-function labels. Duplicate conditions are retained because they represent repeat evaluations. `Desirability_exp` is used only to check agreement with the newly calculated scores; predicted outcomes are not used for baseline evaluation.

Place `237_validation_set.xlsx` and `25_BO_experiments.xlsx` in the same folder as this notebook. Jupyter normally uses that folder as its working directory; the first import cell prints it so you can check. These are the only two Excel imports in the notebook. The sheet names are explicit to avoid accidentally loading the wrong sheet.

In [ ]:
# Replacement for the original Excel-only input layer: use supplied, versioned CSV datasets.
CANDIDATE_FILE = DATA_DIR / 'candidate_measurements.csv'
BO_FILE = DATA_DIR / 'recorded_evaluations.csv'
OUTPUT_DIR = OUTPUT_ROOT / 'experimental_efficiency'
for path in (CANDIDATE_FILE, BO_FILE):
    assert path.is_file(), f'Missing input: {path}'
print('CSV input files:', CANDIDATE_FILE.name, BO_FILE.name)

### Import the 237-condition Excel sheet
**Input:** `237_validation_set.xlsx`, sheet `clean`. No changes are written to this file.

In [ ]:
# Preserve the names and units used in the original published efficiency code.
rename = {'time_h':'time /h','substrate_g_L':'substrate /g  L-1',
'enzyme_nM':'enzyme /Nm','EG_vol_percent':'EG / vol%',
'temperature_C':'T / °C','pH':'ph / -',
'BHET_selectivity_percent':'S_BHET / %',
'BHET_yield_mg_g_PET':'Y_BHET / mgBHET gPET−1'}
candidates = pd.read_csv(CANDIDATE_FILE).rename(columns=rename)
candidates.insert(0, 'Expt Num', np.arange(1, len(candidates)+1))
print(f'Loaded {len(candidates)} candidates.')

### Import the 25-evaluation BO Excel sheet
**Input:** `25_BO_experiments.xlsx`, sheet `MeasuredRecommendations`. Repeat evaluations stay in the table.

In [ ]:
bo = pd.read_csv(BO_FILE).rename(columns={
'cycle':'Cycle','measured_selectivity_percent':'S_BHET / %',
'measured_yield_mg_g_PET':'Y_BHET / mgBHET gPET−1',
'measured_desirability':'Desirability_exp',
**{k:v for k,v in rename.items() if k in ['time_h','substrate_g_L','enzyme_nM','EG_vol_percent','temperature_C','pH']}
})
# Original historical `substrate` counts PET discs (4 discs = 128 g/L).
# The curated screening dataset stores the same loading as g/L; 1 disc = 32 g/L.
bo['substrate /g  L-1'] = bo['substrate /g  L-1'] * 32.0
bo['Expt Num'] = -1  # Populated by exact measured-input matching in the next section.
print(f'Loaded {len(bo)} historical BO evaluation rows.')

## 3. Validate conditions and match BO to the candidate pool

The six inputs identify a condition. Experiment IDs and acquisition-function labels are **not** input dimensions for space-filling. Matching permits tiny floating-point differences in the input values. All measured outcomes must be complete and agree between the two files.

We retain repeated BO rows, but require the 237-condition pool itself to contain distinct conditions.

In [ ]:
FEATURES = ["time /h", "substrate /g  L-1", "enzyme /Nm", "EG / vol%", "T / °C", "ph / -"]
SELECTIVITY = "S_BHET / %"
YIELD = "Y_BHET / mgBHET gPET−1"
ID = "Expt Num"

### Check the input tables
Stop here if required values or unique candidate IDs are missing.

In [ ]:
assert len(candidates) == 237, "Expected 237 candidates."
assert len(bo) == 25, "Expected 25 BO evaluation rows."
assert candidates[ID].is_unique, "Candidate IDs must be unique."
for frame in [candidates, bo]:
    assert not frame[FEATURES + [SELECTIVITY, YIELD]].isna().any().any(), "Missing required values."
    assert np.isfinite(frame[FEATURES + [SELECTIVITY, YIELD]].to_numpy(float)).all()
assert not candidates.duplicated(FEATURES).any(), "Duplicate candidate conditions."

### Match BO conditions to the 237-condition pool
The measured outcomes must agree between files.

In [ ]:
matched_indices = []
for _, row in bo.iterrows():
    match = np.isclose(candidates[FEATURES].to_numpy(float), row[FEATURES].to_numpy(float),
                       atol=1e-8, rtol=1e-8).all(axis=1)
    indices = np.flatnonzero(match)
    assert len(indices) == 1, "Each BO condition must match exactly one candidate."
    index = indices[0]
    assert np.allclose(candidates.iloc[index][[SELECTIVITY, YIELD]].to_numpy(float),
                       row[[SELECTIVITY, YIELD]].to_numpy(float), atol=1e-8, rtol=1e-8)
    matched_indices.append(index)
bo[ID] = candidates.iloc[matched_indices][ID].to_numpy()

In [ ]:
cycle_counts = bo.groupby("Cycle").agg(evaluations=(ID, "size"), unique_conditions=(ID, "nunique"))
print(cycle_counts.to_string())
print(f"Overall: {len(bo)} evaluations, {bo[ID].nunique()} unique conditions.")

## 4. Calculate desirability — measured outcomes only

Equal weights balance the **normalized** selectivity and yield:

$$D = \frac{1}{2}\left(\frac{S_{BHET}(\%)}{100}+\frac{Y_{BHET}(\mathrm{mg\ BHET/g\ PET})}{12.910625}\right).$$

Both lower bounds are zero. All supplied measurements are inside these bounds. The code stops if new data fall outside them rather than silently changing the agreed scoring rule. Original measurements and reported scores are preserved; calculated scores go into a new column.

In [ ]:
SELECTIVITY_UPPER = 100.0
YIELD_UPPER = 12.910625
WEIGHT_SELECTIVITY = 1.0
WEIGHT_YIELD = 1.0

### Define the scoring formula
Selectivity and yield each contribute half of the normalized score.

In [ ]:
def calculate_desirability(frame):
    assert frame[SELECTIVITY].between(0, SELECTIVITY_UPPER).all(), "Selectivity outside bounds."
    assert frame[YIELD].between(0, YIELD_UPPER).all(), "Yield outside bounds."
    return ((frame[SELECTIVITY] / SELECTIVITY_UPPER) * WEIGHT_SELECTIVITY
            + (frame[YIELD] / YIELD_UPPER) * WEIGHT_YIELD) / (WEIGHT_SELECTIVITY + WEIGHT_YIELD)

### Calculate and check the scores

In [ ]:
candidates["Desirability_calculated"] = calculate_desirability(candidates)
bo["Desirability_calculated"] = calculate_desirability(bo)
if "Desirability_exp" in bo.columns:
    difference = abs(bo["Desirability_calculated"] - bo["Desirability_exp"])
    assert difference.max() < 1e-12, "Calculated scores differ from reported BO scores."
    print(f"Largest difference from reported BO scores: {difference.max():.3g}")
print(candidates[[ID, SELECTIVITY, YIELD, "Desirability_calculated"]].head().to_string(index=False))

## 5. Establish the BO target and observed progress

The target is BO's best measured desirability, not its best model prediction. We report BO progress at the end of each batch: ordering rows within one batch does not establish a sequential discovery order.

For this dataset the target is also the best condition in the candidate pool. This does not establish an optimum outside these 237 conditions.

In [ ]:
scores = candidates["Desirability_calculated"].to_numpy()
TARGET = float(bo["Desirability_calculated"].max())
n_candidates = len(candidates)
target_mask = scores >= TARGET - TARGET_TOLERANCE
condition_ids = candidates[ID].to_numpy()


print(f"BO target: {TARGET:.9f}")

### Observed BO progress at the end of each cycle

In [ ]:
progress = []
for cycle in sorted(bo["Cycle"].unique()):
    observed = bo[bo["Cycle"] <= cycle]
    progress.append({"Cycle": int(cycle), "Evaluations": len(observed),
                     "Unique conditions": observed[ID].nunique(),
                     "Best desirability": observed["Desirability_calculated"].max()})
bo_progress = pd.DataFrame(progress)
print(f"Target desirability: {TARGET:.9f}")
print("Qualifying candidate IDs:", condition_ids[target_mask].tolist())
print(bo_progress.to_string(index=False))

## 6. Random sampling — fixed budget of 25 experiments

**Question:** if 25 different conditions are chosen at random, how often do they reach BO's target, and what is the best desirability usually found?

Each campaign starts with a random ordering of all 237 conditions; its first 25 are the fixed-budget sample. Full orderings are retained for the separate sequential analysis below. Reusing them makes the two questions consistent; it does not create two independent sets of evidence.

In [ ]:
random_rng = baseline_rng(0)
random_orders = np.array([random_rng.permutation(n_candidates) for _ in range(N_CAMPAIGNS)], dtype=np.int16)
random_best_by_budget = np.maximum.accumulate(scores[random_orders], axis=1)
random_at_25 = random_best_by_budget[:, BUDGET - 1]
random_success_25 = random_at_25 >= TARGET - TARGET_TOLERANCE

print(f"Success within {BUDGET} evaluations: {random_success_25.mean():.2%}")
print(f"Median best desirability: {np.median(random_at_25):.6f}")
print("Middle 50% of best desirability:", np.quantile(random_at_25, [0.25, 0.75]))

## 7. Random sequential sampling — experiments needed to reach the target

**Question:** when randomly selected conditions are evaluated one at a time, how many evaluations are needed to find the target?

We locate the first qualifying condition in each random ordering. Because the complete candidate pool contains the target, every full campaign eventually succeeds. The median is the middle campaign; the arithmetic mean is the average.

In [ ]:
random_hits = target_mask[random_orders]
assert random_hits.any(axis=1).all(), "A campaign did not contain a qualifying condition."
random_to_target = random_hits.argmax(axis=1) + 1  # Convert zero-based position to experiment count.
assert np.array_equal(random_to_target <= BUDGET, random_success_25)

print(f"Median evaluations to target: {np.median(random_to_target):g}")
print(f"Mean evaluations to target: {random_to_target.mean():.4f}")
print("Middle 50% of evaluations:", np.quantile(random_to_target, [0.25, 0.75]))

# Exact theoretical check for this dataset: one qualifying condition among 237.
if target_mask.sum() == 1:
    exact_probability = BUDGET / n_candidates
    exact_mean_median = (n_candidates + 1) / 2
    print(f"Exact success probability: {exact_probability:.6%}")
    print(f"Exact mean/median evaluations: {exact_mean_median:g}")
    print("Small simulation differences are expected because the number of campaigns is finite.")

## 8. Space-filling sampling — fixed-budget and sequential analysis

**Rule:** start at a randomly chosen condition. At each step, choose the available condition whose nearest previously selected condition is farthest away (sequential **maximin**).

All six experimental inputs are scaled to 0–1 using their candidate-space ranges and given equal geometric weight. This prevents units alone from dominating distance. Yield and selectivity never enter the selection rule. Ties are broken randomly. This is one specified space-filling benchmark, not a conventional factorial DoE.

The campaign-running cell takes the longest to run. Every campaign is a full sequence without replacement, enabling both performance at 25 and experiments-to-target calculations.

In [ ]:
raw_inputs = candidates[FEATURES].to_numpy(float)
input_min = raw_inputs.min(axis=0)
input_max = raw_inputs.max(axis=0)
assert (input_max > input_min).all(), "Remove constant inputs before distance scaling."
scaled_inputs = (raw_inputs - input_min) / (input_max - input_min)

# Squared Euclidean distance has the same ordering as Euclidean distance.
distances = ((scaled_inputs[:, None, :] - scaled_inputs[None, :, :]) ** 2).sum(axis=2)

### Run space-filling campaigns
The next point maximizes its distance to the nearest selected point. This selection uses inputs only.

In [ ]:
space_rng = baseline_rng(1)
space_orders = np.empty((N_CAMPAIGNS, n_candidates), dtype=np.int16)

for campaign in range(N_CAMPAIGNS):
    selected = np.zeros(n_candidates, dtype=bool)
    nearest_distance = np.full(n_candidates, np.inf)
    next_point = int(space_rng.integers(n_candidates))
    for step in range(n_candidates):
        space_orders[campaign, step] = next_point
        selected[next_point] = True
        nearest_distance = np.minimum(nearest_distance, distances[next_point])
        nearest_distance[selected] = -np.inf  # Prevent selecting a condition again.
        if step < n_candidates - 1:
            tied = np.flatnonzero(np.isclose(nearest_distance, nearest_distance.max(), rtol=0, atol=1e-12))
            next_point = int(space_rng.choice(tied))
    if (campaign + 1) % max(1, N_CAMPAIGNS // 4) == 0:
        print(f"Completed {campaign + 1:,} / {N_CAMPAIGNS:,} space-filling campaigns")

### Evaluate space-filling performance
Now reveal measured desirability and calculate both types of result.

In [ ]:
space_best_by_budget = np.maximum.accumulate(scores[space_orders], axis=1)
space_at_25 = space_best_by_budget[:, BUDGET - 1]
space_to_target = target_mask[space_orders].argmax(axis=1) + 1
space_success_25 = space_to_target <= BUDGET
print(f"Success within {BUDGET} evaluations: {space_success_25.mean():.2%}")
print(f"Median best desirability: {np.median(space_at_25):.6f}")
print(f"Median evaluations to target: {np.median(space_to_target):g}")
print(f"Mean evaluations to target: {space_to_target.mean():.4f}")

## 9. Compare campaign variability

The interquartile range (IQR) contains the middle 50% of simulated campaigns. The 2.5th–97.5th percentile range describes the central 95% of campaign outcomes; neither is a confidence interval for the median. Monte Carlo standard error describes the precision of the simulated success probability.

Compare the baselines with the observed BO target reached by 25 additional evaluations. A single observed BO campaign does not provide a BO success probability.

### Build the detailed result tables
This bookkeeping cell collects the already calculated results for export. The compact comparison appears in the final cell.

In [ ]:
summary_rows = []
campaign_frames = []
curve_frames = []
for name, orders, best, hitting in [
    ("Random", random_orders, random_best_by_budget, random_to_target),
    ("Space-filling", space_orders, space_best_by_budget, space_to_target),
]:
    assert np.all(np.sort(orders, axis=1) == np.arange(n_candidates))
    at_budget = best[:, BUDGET - 1]
    p = float((hitting <= BUDGET).mean())
    summary_rows.append({"Strategy": name, "Success probability": p,
        "Probability MC SE": np.sqrt(p * (1-p) / N_CAMPAIGNS),
        "Median best desirability": np.median(at_budget),
        "Median evaluations": np.median(hitting), "Mean evaluations": hitting.mean(),
        "Evaluations Q25": np.quantile(hitting, .25), "Evaluations Q75": np.quantile(hitting, .75),
        "Evaluations Q2.5": np.quantile(hitting, .025), "Evaluations Q97.5": np.quantile(hitting, .975)})
    campaign_frames.append(pd.DataFrame({"Strategy": name, "Campaign": np.arange(1, N_CAMPAIGNS+1),
        "Start condition ID": condition_ids[orders[:, 0]], "Best desirability at budget": at_budget,
        "Reached target at budget": hitting <= BUDGET, "Evaluations to target": hitting}))
    quantiles = np.quantile(best, [.025, .25, .5, .75, .975], axis=0)
    curve_frames.append(pd.DataFrame({"Strategy": name, "Evaluations": np.arange(1, n_candidates+1),
        "Success probability": [(hitting <= k).mean() for k in range(1, n_candidates+1)],
        "Best Q2.5": quantiles[0], "Best Q25": quantiles[1], "Best median": quantiles[2],
        "Best Q75": quantiles[3], "Best Q97.5": quantiles[4]}))
summary = pd.DataFrame(summary_rows)
campaign_results = pd.concat(campaign_frames, ignore_index=True)
performance_curves = pd.concat(curve_frames, ignore_index=True)
print(summary.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print(f"Observed BO: target {TARGET:.6f} by cycle 5 / 25 evaluations (18 distinct conditions).")

## 10. Figures displayed in Jupyter

Each figure has its own cell and ends with `plt.show()`. Install Matplotlib with `%pip install matplotlib` if needed, then rerun the plotting cells. The calculations and final summary also work without Matplotlib.

In [ ]:
# Matplotlib is optional; every figure below is displayed with plt.show().
figures = {}
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Install matplotlib to display figures: %pip install matplotlib")

### Figure 1 — best desirability
Lines show baseline medians; shading shows the middle 50% of campaigns. BO points are observed batch endpoints.

In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    for name, color in [("Random", "#4267A1"), ("Space-filling", "#C47B27")]:
        curve = performance_curves[performance_curves["Strategy"] == name]
        x = curve["Evaluations"].to_numpy()
        ax.plot(x, curve["Best median"], color=color, label=name)
        ax.fill_between(x, curve["Best Q25"].to_numpy(), curve["Best Q75"].to_numpy(), color=color, alpha=.18)
    ax.plot(bo_progress["Evaluations"], bo_progress["Best desirability"], "o-", color="black", label="Observed BO")
    ax.axhline(TARGET, color="gray", linestyle=":", label="BO target")
    ax.set(xlabel="Experimental evaluations", ylabel="Best desirability", xlim=(1, 60))
    ax.legend(frameon=False)
    figures["best_desirability"] = fig
    plt.show()

### Figure 2 — probability of reaching the target
Random sequential sampling uses the same random campaigns as the fixed-budget analysis. The vertical line marks 25 evaluations.

In [ ]:
if plt is not None:
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    for name, label in [("Random", "Random sequential sampling"), ("Space-filling", "Space-filling")]:
        curve = performance_curves[performance_curves["Strategy"] == name]
        ax.plot(curve["Evaluations"], curve["Success probability"], label=label)
    ax.axvline(BUDGET, color="gray", linestyle=":", label=f"{BUDGET} evaluations")
    ax.set(xlabel="Experimental evaluations", ylabel="Probability of reaching BO target", xlim=(1, n_candidates), ylim=(0, 1))
    ax.legend(frameon=False)
    figures["target_probability"] = fig
    plt.show()

## 11. Save prepared data and results

This cell writes CSV tables and a JSON method record to `OUTPUT_DIR`. It also saves complete campaign selection orders for reproducibility. CSV files open in Excel. Original input workbooks are not modified. Rerunning replaces only these derived output files in the chosen output folder.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
candidates.to_csv(OUTPUT_DIR / "237_candidates_with_desirability.csv", index=False)
bo.to_csv(OUTPUT_DIR / "25_bo_evaluations_with_desirability.csv", index=False)
summary.to_csv(OUTPUT_DIR / "baseline_summary.csv", index=False)
campaign_results.to_csv(OUTPUT_DIR / "campaign_results.csv", index=False)
performance_curves.to_csv(OUTPUT_DIR / "performance_by_budget.csv", index=False)
bo_progress.to_csv(OUTPUT_DIR / "observed_bo_by_cycle.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "campaign_condition_orders.npz",
                    random=condition_ids[random_orders], space_filling=condition_ids[space_orders])
method = {"seed": SEED, "campaigns_per_baseline": N_CAMPAIGNS, "budget": BUDGET,
          "candidate_file": str(CANDIDATE_FILE), "bo_file": str(BO_FILE),
          "target": TARGET, "selectivity_upper": SELECTIVITY_UPPER, "yield_upper": YIELD_UPPER,
          "weights": [WEIGHT_SELECTIVITY, WEIGHT_YIELD], "sampling": "without replacement",
          "space_filling": "min-max scaled, equal input weights, sequential maximin, random start and ties",
          "scaling": {f: [float(lo), float(hi)] for f, lo, hi in zip(FEATURES, input_min, input_max)},
          "numpy_version": np.__version__, "pandas_version": pd.__version__,
          "interpretation": "Additional evaluations after historical BO training; fixed measured means; observed BO only."}
(OUTPUT_DIR / "method.json").write_text(json.dumps(method, indent=2), encoding="utf-8")
for name, fig in figures.items():
    fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{name}.pdf", bbox_inches="tight")
print("Results saved to:", OUTPUT_DIR)

## What the results support

- Compare the observed BO campaign with the distributions of baseline outcomes under the same **additional-evaluation budget**.
- Report both the chance of reaching the target by 25 and the median/IQR of evaluations to target.
- Space-filling can be competitive even when its median target-reaching count exceeds BO's 25 evaluations.
- The exact target is based on fixed measured means. No experimental-equivalence tolerance or measurement-noise model was introduced.
- Do not interpret baseline campaign variability as variability between BO runs, or the 25 evaluations as including the historical training cost.

Changing weights or normalization bounds changes the evaluation criterion. It does not regenerate the historical BO recommendations.

## 12. Main results — final summary DataFrame

The table below follows the requested layout. **Average** means the arithmetic mean; **median** means the middle campaign.

Random sampling is the fixed-budget question, so evaluations-to-target entries are **Not applicable**. The separate random-sequential column reports those counts. Its first two rows match random sampling because both examine the same random sequences at evaluation 25. They are two views of one strategy, not independent benchmarks.

In [ ]:
# Helper for a readable middle-50% range.
def middle_half(values):
    lower, upper = np.quantile(values, [0.25, 0.75])
    return f"{lower:g}–{upper:g}"

measures = [
    f"Probability of reaching target within {BUDGET} evaluations",
    f"Median best desirability after {BUDGET} evaluations",
    "Median evaluations to reach target",
    "Average evaluations to reach target",
    "Middle 50% of evaluations to target",
]

In [ ]:
summary_df = pd.DataFrame({
    "Measure": measures,
    "Random sampling": [
        f"{random_success_25.mean():.2%}",
        f"{np.median(random_at_25):.6f}",
        "Not applicable", "Not applicable", "Not applicable",
    ],
    "Random sequential sampling": [
        f"{random_success_25.mean():.2%}",
        f"{np.median(random_at_25):.6f}",
        f"{np.median(random_to_target):g}",
        f"{random_to_target.mean():.2f}",
        middle_half(random_to_target),
    ],
    "Space-filling": [
        f"{space_success_25.mean():.2%}",
        f"{np.median(space_at_25):.6f}",
        f"{np.median(space_to_target):g}",
        f"{space_to_target.mean():.2f}",
        middle_half(space_to_target),
    ],
}).set_index("Measure")

#summary_df.to_csv(OUTPUT_DIR / "main_results_summary.csv")
#pd.set_option("display.max_colwidth", None)
summary_df  # Jupyter displays the DataFrame as a table.

### Manuscript Table 2 validation and corrected additional-effort comparison

**Reported Table 2 (Monte Carlo reference):** random success 10.15%, mean 119.76, median 121; space-filling success 42.48%, mean 55.6, median 38. The assertions below will flag any algorithm/input change that fails to recreate the reported results. The article's original claims of “6×” and “2.5×” do not follow from the stated 25-row BO budget and Table 2 means; the correct nominal ratios are approximately 4.79× and 2.22×.

The assertions are deliberately strict for a fixed-version environment. If they fail after a library change, **investigate** rather than silently changing results.

In [ ]:
# Exact reproducibility checks against manuscript Table 2.
assert np.isclose(random_success_25.mean(), .1015, atol=1e-12), 'Random fixed-budget probability differs from Table 2.'
assert np.isclose(random_to_target.mean(), 119.7552, atol=1e-4), 'Random mean differs from the original notebook.'
assert np.median(random_to_target) == 121, 'Random median differs from Table 2.'
assert np.isclose(space_success_25.mean(), .4248, atol=1e-12), 'Space-filling probability differs from Table 2.'
assert np.isclose(space_to_target.mean(), 55.6006, atol=1e-4), 'Space-filling mean differs from the original notebook.'
assert np.median(space_to_target) == 38, 'Space-filling median differs from Table 2.'
comparison = pd.DataFrame({
 'Strategy':['Historical BO','Random screening','Space-filling screening'],
 'Mean additional evaluation rows':[BUDGET,random_to_target.mean(),space_to_target.mean()]})
comparison['Relative evaluations vs BO'] = comparison['Mean additional evaluation rows']/BUDGET
comparison['Budget reduction using BO (%)'] = (1-BUDGET/comparison['Mean additional evaluation rows'])*100
print('All manuscript Table 2 values reproduced under the provided inputs/settings.')
display(comparison)
comparison.to_csv(OUTPUT_DIR/'manuscript_efficiency_comparison.csv',index=False)

---
## Scientific interpretation and limitations

- **Historical campaign:** The documented campaign selects five evaluation rows per cycle, with qLogEI/qPI recommendations and duplicate conditions retained. Best-so-far values can differ from per-cycle maxima. 
- **Robustness:** Monte Carlo results depend on initial size/composition, seeds, acquisition settings, and BayBE/BoTorch versions. They must be regenerated in the pinned environment before claiming the 180-run study was independently reproduced.
- **Screening efficiency:** Baselines are **retrospective lookups** over 237 measured conditions, with random/space-filling selection ignoring measured responses until a condition is sampled. The 25 BO evaluation rows represent 18 distinct measured conditions, whereas standalone baselines draw unique conditions without replacement.
- **Target:** D is based on equal-weight normalized yield and selectivity, with bounds 12.910625 mg/g PET and 100%. A fixed target is derived from the historical BO observed maximum.
- **Manuscript Table 2:** Monte Carlo mean/budget comparisons are 119.76/25 ≈ 4.79 and 55.6/25 ≈ 2.22. These are comparisons of *additional nominal evaluations* after the initial 203-point dataset, not end-to-end experiment totals.
- **Reproducibility:** The historical replay can vary with package versions. Inputs, core measurement data, original scripts, and a dependency record should be archived with the paper. No data were fabricated or fitted to manuscript numbers.
